# CamoPatch B-cos Kaggle Job

Edit `JOB_CONFIG` for a single matrix condition, then run the next cell. The scheduler writes the same `job_config.json` automatically.

In [ ]:
import json
from pathlib import Path

JOB_CONFIG = {
    "job_id": "camopatch-bcos-resnet50-s16-linf16_256-bcos_top1-smoke",
    "attack": "camopatch",
    "model_family": "bcos",
    "model": "resnet50",
    "patch_size": 16,
    "linf": "16/256",
    "position": "bcos_top1",
    "queries": 10000,
    "images_csv": "data/used_images_1000.csv",
    "image_batch_size": 1000,
    "device": "cuda",
    "fixed_position": True,
    "save_images": False,
    "code_dataset_owner": "hkhnhduy",
    "code_dataset_slug": "attack-bcos-github",
    "github_repo": "https://github.com/voicon324/attack_bcos.git",
    "github_ref": "main",
}

Path("job_config.json").write_text(json.dumps(JOB_CONFIG, indent=2) + "\n")
JOB_CONFIG

In [ ]:
import subprocess
import sys
from pathlib import Path

runner_candidates = [
    Path("/kaggle/input/attack-bcos-github/kaggle/camopatch_job/run_camopatch_job.py"),
    Path("/kaggle/input/datasets/hkhnhduy/attack-bcos-github/kaggle/camopatch_job/run_camopatch_job.py"),
    Path("kaggle/camopatch_job/run_camopatch_job.py"),
]
runner = next((path for path in runner_candidates if path.is_file()), None)
if runner is None:
    for path in Path("/kaggle/input").rglob("kaggle/camopatch_job/run_camopatch_job.py"):
        runner = path
        break
if runner is None:
    repo_dir = Path("/kaggle/working/attack_bcos")
    if repo_dir.exists():
        subprocess.check_call(["rm", "-rf", str(repo_dir)])
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", JOB_CONFIG.get("github_ref", "main"), JOB_CONFIG.get("github_repo", "https://github.com/voicon324/attack_bcos.git"), str(repo_dir)])
    runner = repo_dir / "kaggle/camopatch_job/run_camopatch_job.py"

subprocess.check_call([sys.executable, "-u", str(runner), "--config", "job_config.json"], cwd=str(runner.parents[2]))
